<a href="https://colab.research.google.com/github/TiferetChayahShalom/AI_Document_Search_Summarization/blob/main/AI_Document_Search_Summarization_Tiferet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**AI-Powered Document Search and Summarization System**



Objective:
Build an AI-driven tool to ingest documents, perform semantic search, and generate concise summaries of relevant sections.



Step-by-Step Instructions
1. Define Scope & Document Types

First, decide which document formats to support (PDF, Word, plain text).
Then, outline a high-level ingestion workflow:


In [132]:
!pip -q install pymupdf python-docx sentence-transformers faiss-cpu transformers sentencepiece rouge-score nltk

2. Implement Document Ingestion & Text Extraction



In [133]:
#i chosed plain text
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
  print(filename)

Saving sample.txt to sample (2).txt
Uploaded files:
sample (2).txt


In [134]:
import os
import fitz # PyMuPDF
from docx import Document

documents = []

def extract_text(filename):
  extension = os.path.splitext(filename)[1].lower()

  if extension == ".pdf":
    text_parts = []
    with fitz.open(filename) as pdf:

      for page in pdf:
        text_parts.append(page.get_text())
    return "\n".join(text_parts)

  elif extension == ".docx":
    doc = Document(filename)
    return "\n".join(p.text for p in doc.paragraphs)

  elif extension == ".txt":
    with open(filename, "r", encoding="utf-8-sig") as file:
      return file.read()

  else:
    print(f"Skipping unsupported file: {filename}")
    return None



for filename in uploaded:
  text = extract_text(filename)

  if text and text.strip():
    documents.append({
    "filename": filename,
    "text": text.strip()
    })
    print(f"Extracted: {filename} ({len(text)} characters)")

print(f"\nTotal documents extracted: {len(documents)}")

Extracted: sample (2).txt (427 characters)

Total documents extracted: 1


3. Split & Preprocess Text

First, segment the extracted text into manageable chunks (e.g., 200–500 tokens per chunk).
Then, clean and normalize each chunk (remove headers, footers, whitespace).


In [135]:
chunks = []

CHUNK_SIZE = 300 # words per chunk
OVERLAP = 50 # words shared between chunks

for doc in documents:
  words = doc["text"].split()
  start = 0
  chunk_index = 0

  while start < len(words):
    end = min(start + CHUNK_SIZE, len(words))
    chunk_text = " ".join(words[start:end])

    chunks.append({
      "filename": doc["filename"],
      "chunk_index": chunk_index,
      "text": chunk_text
      })

    chunk_index += 1

    if end == len(words):
      break

    start = end - OVERLAP

print("Total chunks created:", len(chunks))

if chunks:
  print("\nFirst chunk:")
  print(chunks[0]["text"][:500])

Total chunks created: 1

First chunk:
Artificial Intelligence in Education Artificial intelligence can help students learn more effectively. AI tools can explain difficult concepts, answer questions, and provide personalized feedback. Teachers can use AI to summarize educational materials and find relevant information in large collections of documents. However, AI-generated answers should be checked for accuracy, and personal information should be protected.


4. Generate Embeddings

First, load a sentence-transformer model (e.g., all-MiniLM-L6-v2).
Then, compute embeddings for each text chunk.


In [136]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedding_model = SentenceTransformer(
  "sentence-transformers/all-MiniLM-L6-v2",
  device="cpu"
)

texts = [chunk["text"] for chunk in chunks]

if not texts:
  raise ValueError("No text chunks found. Complete Steps 2–5 first.")

embeddings = embedding_model.encode(
  texts,
  batch_size=4,
  show_progress_bar=True,
  convert_to_numpy=True,
  normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)
print("Number of embedded chunks:", len(embeddings))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (1, 384)
Number of embedded chunks: 1


5. Build & Populate the Vector Database

First, initialize FAISS or Pinecone index with CPU-friendly settings.
Then, insert your chunk embeddings and keep metadata (document ID, chunk index).



In [137]:
import faiss
import numpy as np

# 1. Prepare the chunk embeddings for FAISS
embeddings = np.asarray(embeddings, dtype="float32")

# 2. Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

# 3. Initialize a CPU-friendly FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# 4. Insert all chunk embeddings into the index
index.add(embeddings)

# 5. Store metadata for each chunk
chunk_metadata = []

for doc_id, chunk in enumerate(chunks):
  chunk_metadata.append({
  "document_id": chunk["filename"],
  "chunk_index": chunk["chunk_index"],
  "text": chunk["text"]
  })

# 6. Verify the index and metadata
print("FAISS index created successfully!")
print("Embedding vectors stored:", index.ntotal)
print("Metadata records stored:", len(chunk_metadata))
print("Embedding dimension:", dimension)

FAISS index created successfully!
Embedding vectors stored: 1
Metadata records stored: 1
Embedding dimension: 384


6. Create the Search Interface

First, build a query input form in your app.
Then, on query submission: embed the query, search the vector store for top‑k chunks, and return the most relevant ones.


In [138]:
def search_documents(query, top_k=3):
  # Convert the question into an embedding
  query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
  ).astype("float32")

  # Search the FAISS index
  k = min(top_k, index.ntotal)
  scores, indices = index.search(query_embedding, k)

  # Return the most relevant passages
  results = []

  for score, idx in zip(scores[0], indices[0]):
      if idx == -1:
          continue

      results.append({
        "document_id": chunk_metadata[idx]["document_id"],
        "chunk_index": chunk_metadata[idx]["chunk_index"],
        "text": chunk_metadata[idx]["text"],
        "score": float(score)
      })

  return results


# Test the search
query = "How can artificial intelligence help students?"
results = search_documents(query, top_k=3)

for i, result in enumerate(results, start=1):
  print(f"\nResult {i}")
  print("Document:", result["document_id"])
  print("Similarity:", round(result["score"], 3))
  print(result["text"])


Result 1
Document: sample (2).txt
Similarity: 0.724
Artificial Intelligence in Education Artificial intelligence can help students learn more effectively. AI tools can explain difficult concepts, answer questions, and provide personalized feedback. Teachers can use AI to summarize educational materials and find relevant information in large collections of documents. However, AI-generated answers should be checked for accuracy, and personal information should be protected.


7. Summarize Retrieved Content

First, load a summarization model (e.g., BART-base or t5-small).
Then, pass the concatenated top‑k chunks into the summarizer to produce a concise summary.

In [139]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

summarizer_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(summarizer_name)
summarizer = AutoModelForSeq2SeqLM.from_pretrained(
  summarizer_name
)
summarizer.to("cpu")
summarizer.eval()



def summarize_results(results):
  if not results:
    return "No relevant passages were found."

  context = "\n\n".join(
    result["text"] for result in results
  )

  # Limit input size to fit the model
  inputs = tokenizer(
      "summarize: " + context,
      return_tensors="pt",
      max_length=512,
      truncation=True
  )

  outputs = summarizer.generate(
    **inputs,
    max_new_tokens=120,
    num_beams=2,
    do_sample=False
  )

  return tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
  )



summary = summarize_results(results)

print("SUMMARY")
print(summary)

print("\nSOURCES")
for result in results:
  print(
      f"- {result['document_id']}, "
      f"chunk {result['chunk_index']}"
  )

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

SUMMARY
artificial intelligence can explain difficult concepts, answer questions, and provide personalized feedback. teachers can use AI to summarize educational materials and find relevant information in large collections of documents. however, AI-generated answers should be checked for accuracy.

SOURCES
- sample (2).txt, chunk 0


8. Evaluate Search & Summarization

First, assemble a test set of queries with ground‑truth relevant passages and summaries.
Then, compute:
Search: precision@k, recall@k
Summarization: BLEU, ROUGE, perplexity


In [140]:
!pip -q install rouge-score nltk

In [141]:
for i, chunk in enumerate(chunk_metadata):
  print(f"\nIndex: {i}")
  print("Document:", chunk["document_id"])
  print("Chunk:", chunk["chunk_index"])
  print("Text:", chunk["text"][:300])


Index: 0
Document: sample (2).txt
Chunk: 0
Text: Artificial Intelligence in Education Artificial intelligence can help students learn more effectively. AI tools can explain difficult concepts, answer questions, and provide personalized feedback. Teachers can use AI to summarize educational materials and find relevant information in large collectio


In [142]:
def evaluate_search(test_queries, k=3):
  precision_values = []
  recall_values = []

  chunk_lookup = {
    (c["document_id"], c["chunk_index"]): i
    for i, c in enumerate(chunk_metadata)
  }

  for test in test_queries:
    relevant = test["relevant_indices"]
    results = search_documents(test["query"], top_k=k)

    retrieved_indices = {
    chunk_lookup[(r["document_id"], r["chunk_index"])]
    for r in results
    if (r["document_id"], r["chunk_index"]) in chunk_lookup
  }

    hits = len(retrieved_indices & relevant)

    precision = (hits / len(retrieved_indices)if retrieved_indices else 0)
    recall = hits / len(relevant) if relevant else 0

    precision_values.append(precision)
    recall_values.append(recall)

    print(f"\nQuery: {test['query']}")
    print("Retrieved indices:", retrieved_indices)
    print("Number retrieved:", len(retrieved_indices))
    print("Retrieved chunks:", retrieved_indices)
    print("Relevant chunks:", relevant)
    print(f"Precision@{k}: {precision:.3f}")
    print(f"Recall@{k}: {recall:.3f}")

  if precision_values:
    print(
    "\nAverage Precision@k:",
    round(sum(precision_values) / len(precision_values), 3)
  )
    print(
    "Average Recall@k:",
    round(sum(recall_values) / len(recall_values), 3)
  )



test_queries = [
  {
    "query": "What are the main findings?",
    "relevant_indices": {0}
  },
  {
    "query": "What recommendations are given?",
    "relevant_indices": {0}
  }
]

evaluate_search(test_queries, k=3)


Query: What are the main findings?
Retrieved indices: {0}
Number retrieved: 1
Retrieved chunks: {0}
Relevant chunks: {0}
Precision@3: 1.000
Recall@3: 1.000

Query: What recommendations are given?
Retrieved indices: {0}
Number retrieved: 1
Retrieved chunks: {0}
Relevant chunks: {0}
Precision@3: 1.000
Recall@3: 1.000

Average Precision@k: 1.0
Average Recall@k: 1.0


**What does this mean?**
Precision@3 = 1.00: All retrieved chunks were marked relevant.
Recall@3 = 1.00: All chunks marked relevant were retrieved.
Average Precision and Recall = 1.00: Both questions achieved perfect scores under current evaluation labels.


In [143]:
#Evaluate summarization with BLEU and ROUGE

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Replace this example with a real reference summary
# written from your document.
reference_summary = """
Artificial intelligence can improve education by explaining difficult
concepts, answering questions, and providing personalized feedback.
Teachers can use AI to summarize materials and find relevant information.
However, AI-generated answers should be checked for accuracy, and
personal information must be protected.
"""

generated_summary = summary

scorer = rouge_scorer.RougeScorer(
  ["rouge1", "rougeL"],
  use_stemmer=True
)

rouge_scores = scorer.score(
  reference_summary,
  generated_summary
)

smooth = SmoothingFunction().method1

bleu_score = sentence_bleu(
  [reference_summary.split()],
  generated_summary.split(),
  smoothing_function=smooth
)

print("BLEU:", round(bleu_score, 4))

for metric, values in rouge_scores.items():
  print(
    metric,
    "F1:",
    round(values.fmeasure, 4)
  )

BLEU: 0.3119
rouge1 F1: 0.8354
rougeL F1: 0.8101


In [144]:
# Get a temporary relevant passage from your working search function
results = search_documents(
  "How can artificial intelligence help students?",
  top_k=3
)

if results:
  generated_summary = results[0]["text"]
else:
  generated_summary = ""

print("Generated text:")
print(generated_summary)

Generated text:
Artificial Intelligence in Education Artificial intelligence can help students learn more effectively. AI tools can explain difficult concepts, answer questions, and provide personalized feedback. Teachers can use AI to summarize educational materials and find relevant information in large collections of documents. However, AI-generated answers should be checked for accuracy, and personal information should be protected.


In [145]:
print("Generated summary:")
print(generated_summary)

print("\nReference summary:")
print(reference_summary)

Generated summary:
Artificial Intelligence in Education Artificial intelligence can help students learn more effectively. AI tools can explain difficult concepts, answer questions, and provide personalized feedback. Teachers can use AI to summarize educational materials and find relevant information in large collections of documents. However, AI-generated answers should be checked for accuracy, and personal information should be protected.

Reference summary:

Artificial intelligence can improve education by explaining difficult
concepts, answering questions, and providing personalized feedback.
Teachers can use AI to summarize materials and find relevant information.
However, AI-generated answers should be checked for accuracy, and
personal information must be protected.



Perplexity measures how well a language model predicts a reference sequence. It is not directly comparable across different models or tokenizers, and it is not a substitute for checking summary accuracy. For a consistent reference-summary evaluation, you can calculate T5's token-level loss and its exponent.

In [146]:
import math
import torch

def reference_perplexity(source_text, reference_text):
  inputs = tokenizer(
    "summarize: " + source_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
  )

  labels = tokenizer(
    text_target=reference_text,
    return_tensors="pt",
    max_length=128,
    truncation=True
  )["input_ids"]

  with torch.no_grad():
    output = summarizer(
    **inputs,
    labels=labels
  )

  loss = output.loss.item()

  # Avoid numerical overflow
  return math.exp(min(loss, 20))



if results:
  source_text = "\n\n".join(r["text"] for r in results)

  print(
    "Reference-summary perplexity:",
    round(
    reference_perplexity(
    source_text,
    reference_summary
    ),
    3
    )
  )


Reference-summary perplexity: 2.883


**note:**This computes reference-summary perplexity under the T5 model. Use it as a supplementary metric, not a stand-alone quality score.

**Summarization Evaluation Results**

The T5-small summarization model was evaluated using BLEU, ROUGE-1 F1, ROUGE-L F1, and reference-summary perplexity.

Metric| Result
BLEU| 0.3119
ROUGE-1 F1| 0.8354
ROUGE-L F1| 0.8101
Reference-summary perplexity| 2.883

Interpretation: The ROUGE scores indicate substantial word overlap and sequence similarity between the generated summary and the reference summary. The BLEU score reflects n-gram overlap. The perplexity value measures the model's predictive fit on the evaluated reference text; it should not be treated as a direct measure of summary quality.

These results are based on the current example and should be validated on more documents before drawing broader conclusions.

9. Optimize for CPU & Scale

First, ensure batch embedding and summarization with small batch sizes (2–4).
Then, disable large FAISS features (e.g., IVF) and subsample documents if needed.


In [147]:
import faiss
import numpy as np

embeddings = np.asarray(embeddings, dtype="float32")
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS CPU index created!")
print("Indexed vectors:", index.ntotal)
print("Embedding dimension:", dimension)

FAISS CPU index created!
Indexed vectors: 1
Embedding dimension: 384


In [148]:
print("Embedding batch size: 4")
print("FAISS index type:", type(index).__name__)
print("Indexed vectors:", index.ntotal)

test_results = search_documents(
  "How can AI help students?",
  top_k=3
)

print("Search results:", len(test_results))

if test_results:
  print("Search test passed!")

Embedding batch size: 4
FAISS index type: IndexFlatIP
Indexed vectors: 1
Search results: 1
Search test passed!


10. (Bonus) Automate New Document Processing

First, set up a file-watcher or message queue to detect new uploads.
Then, trigger your ingestion → embedding → indexing pipeline automatically.

In [149]:
!pip -q install watchdog

In [150]:
from pathlib import Path

WATCH_FOLDER = Path("/content/incoming_documents")
WATCH_FOLDER.mkdir(parents=True, exist_ok=True)

print("Folder created:", WATCH_FOLDER)
print("Ready to monitor for new documents.")

Folder created: /content/incoming_documents
Ready to monitor for new documents.


In [151]:
import time
from pathlib import Path
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

def process_new_document(file_path):
  file_path = Path(file_path)

  # Wait until the file has finished copying
  previous_size = -1
  for _ in range(10):
      time.sleep(1)
      if not file_path.exists():
        return
      current_size = file_path.stat().st_size
      if current_size == previous_size:
        break
      previous_size = current_size

  try:
      # Extract text based on file type
      if file_path.suffix.lower() == ".txt":
       text = file_path.read_text(encoding="utf-8", errors="ignore")
      elif file_path.suffix.lower() == ".pdf":
         import fitz
         with fitz.open(file_path) as pdf:
            text = "\n".join(page.get_text() for page in pdf)
      elif file_path.suffix.lower() == ".docx":
          from docx import Document
          doc = Document(file_path)
          text = "\n".join(p.text for p in doc.paragraphs)
      else:
        print("Unsupported file type:", file_path.name)
        return
      if not text.strip():
        print("No text extracted from:", file_path.name)
        return

      # Split text into small chunks
      words = text.split()
      chunk_size = 150
      new_chunks = []

      for start in range(0, len(words), chunk_size):
          part = " ".join(words[start:start + chunk_size])
          new_chunks.append({
          "filename": file_path.name,
          "chunk_index": len(new_chunks),
          "text": part
         })

      # Add the new chunks to the existing collection
      existing_names = {
      c.get("filename", c.get("document_id"))
        for c in chunks
      }

      if file_path.name in existing_names:
        print("This filename is already indexed:", file_path.name)
        return

      chunks.extend(new_chunks)

      # Rebuild embeddings and the FAISS index
      texts = [c["text"] for c in chunks]
      embeddings = embedding_model.encode(
        texts,
        batch_size=4,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
      ).astype("float32")

      import faiss
      index_new = faiss.IndexFlatIP(embeddings.shape[1])
      index_new.add(embeddings)

      metadata_new = [
          {
              "document_id": c.get("filename", c.get("document_id")),
              "chunk_index": c["chunk_index"],
              "text": c["text"]
          }
          for c in chunks
      ]

      # Update the existing notebook variables
      globals()["index"] = index_new
      globals()["chunk_metadata"] = metadata_new

      print(f"Processed and indexed: {file_path.name}")
      print("Total indexed chunks:", index.ntotal)

  except Exception as e:
      print(f"Could not process {file_path.name}: {e}")


class NewDocumentHandler(FileSystemEventHandler):
  def on_created(self, event):
      if not event.is_directory: process_new_document(event.src_path)

handler = NewDocumentHandler()
observer = Observer()
observer.schedule(handler, str(WATCH_FOLDER), recursive=False)
observer.start()

print("Watching for new documents and indexing them automatically.")

Watching for new documents and indexing them automatically.


In [152]:
#test_file creation cell

test_file = WATCH_FOLDER / "watchdog_test.txt"
test_file.write_text(
  "New documents can be processed and indexed automatically.",
  encoding="utf-8"
)
print("Test file created:", test_file.name)

Test file created: watchdog_test.txt


In [153]:
test_file = WATCH_FOLDER / "watchdog_test.txt"

print("File exists:", test_file.exists())

if test_file.exists():
  process_new_document(test_file)
else:
  print("Test file not found. Run the test-file creation cell again.")

File exists: True
Processed and indexed: watchdog_test.txt
Total indexed chunks: 2


In [154]:
test_file = WATCH_FOLDER / "watchdog_test.txt"

test_file.write_text(
  "Artificial intelligence helps students learn. "
  "Teachers can use AI to summarize documents and "
  "find relevant information.",
  encoding="utf-8"
)

print("Test file created:", test_file.name)

Test file created: watchdog_test.txt


In [155]:
results = search_documents(
  "How can AI help students learn?",
  top_k=3
)

print("Search results:", len(results))

for result in results:
  print("\nSource:", result["document_id"])
  print("Chunk:", result["chunk_index"])
  print("Text:", result["text"][:200])

Search results: 2

Source: sample (2).txt
Chunk: 0
Text: Artificial Intelligence in Education Artificial intelligence can help students learn more effectively. AI tools can explain difficult concepts, answer questions, and provide personalized feedback. Tea

Source: watchdog_test.txt
Chunk: 0
Text: New documents can be processed and indexed automatically.


**Automatic Document Processing**

Watchdog was integrated to monitor an incoming-documents folder for new TXT, PDF, and DOCX files. The processing pipeline extracts text, divides it into chunks, generates embeddings in small batches, and updates the FAISS index. A test document was successfully added to the indexed collection and retrieved through a search query. This demonstrates document indexing and retrieval; automatic event-triggered processing should also be tested separately to confirm the complete Watchdog workflow.

11. Document & Reflect

AI-Powered Document Search and Summarization System

1. System Architecture

The system accepts PDF, Word, and TXT documents and extracts their text using document-processing libraries. The text is cleaned and divided into smaller chunks. The "sentence-transformers/all-MiniLM-L6-v2" model converts each chunk into an embedding. FAISS, using "IndexFlatIP" on the CPU, stores and searches these vectors to retrieve relevant passages for a user's query. The retrieved passages are passed to the T5-small model from Hugging Face Transformers, which generates a concise summary. The system displays the summary and its source document.

2. Technology Choices

- Ingestion: PyMuPDF, python-docx, and Python text processing.
- Embeddings: Sentence Transformers with all-MiniLM-L6-v2.
- Vector search: FAISS CPU with IndexFlatIP.
- Summarization: T5-small.
- Evaluation: Precision@k, Recall@k, BLEU, ROUGE-1 F1, ROUGE-L F1, and perplexity.
- Optimization: Small embedding batches, CPU inference, and avoiding unnecessary large-scale indexing features.
- Bonus automation: Watchdog to detect new files and trigger document processing.

3. Evaluation Results

- BLEU: 0.3119
- ROUGE-1 F1: 0.8354
- ROUGE-L F1: 0.8101
- Reference-summary perplexity: 2.883

These results were obtained using the current example. They do not establish performance across a large collection of documents.

4. Reflection

The system demonstrates how semantic search can retrieve document passages and how a pretrained language model can summarize the retrieved information. ROUGE measures overlap with a reference summary, BLEU measures n-gram overlap, and perplexity measures predictive fit under the evaluation setup. Search precision and recall must also be checked using correctly labeled relevant passages.

Small batches and CPU-based models make the system accessible in Google Colab, although processing may be slower as the document collection grows. FAISS IndexFlatIP is simple and suitable for small collections, but searches become more expensive as the number of vectors increases. Testing additional documents and queries would help measure retrieval accuracy and summary quality more reliably.

Watchdog could automate new-document processing while the notebook is running. A persistent deployment would need a continuously running service and durable document storage.


In [156]:
!git --version

git version 2.43.0
